In [1]:
import torch
from diffusion.architectures.classifiers.mnist_classifier import ClassifierTrainer
from whar_datasets.support.getter import WHARDatasetID, get_dataset_cfg
from diffusion.sampleables.whar_sampleable import WHARSampleable, TrainValTest
from diffusion.architectures.classifiers.wisdm_classifier import WISDMClassifier

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
cfg = get_dataset_cfg(WHARDatasetID.WISDM)

In [4]:
sampeable = WHARSampleable(
    cfg=cfg,
    scv_group_index=0,
    fold=TrainValTest.TRAIN,
    # transform=lambda x: x.unsqueeze(0),
)

val_sampeable = WHARSampleable(
    cfg=cfg,
    scv_group_index=0,
    fold=TrainValTest.VAL,
    # transform=lambda x: x.unsqueeze(0),
)

2025-10-22 11:03:06,510 - whar-datasets - INFO - Running DownloadingStep
2025-10-22 11:03:06,510 - whar-datasets - INFO - Checking hash for DownloadingStep
2025-10-22 11:03:06,511 - whar-datasets - INFO - Hash is up to date
2025-10-22 11:03:06,511 - whar-datasets - INFO - Running ParsingStep
2025-10-22 11:03:06,511 - whar-datasets - INFO - Checking hash for ParsingStep
2025-10-22 11:03:06,513 - whar-datasets - INFO - Hash is up to date
2025-10-22 11:03:06,513 - whar-datasets - INFO - Running WindowingStep
2025-10-22 11:03:06,513 - whar-datasets - INFO - Checking hash for WindowingStep
2025-10-22 11:03:06,514 - whar-datasets - INFO - Hash is up to date
2025-10-22 11:03:06,514 - whar-datasets - INFO - Loading windowing
2025-10-22 11:03:06,556 - whar-datasets - INFO - activity_ids from 0 to 5
2025-10-22 11:03:06,556 - whar-datasets - INFO - subject_ids from 0 to 35
2025-10-22 11:03:06,558 - whar-datasets - INFO - train: 17915 | test: 4493
2025-10-22 11:03:06,559 - whar-datasets - INFO - R

In [5]:
print(sampeable.shape)

(6, 32, 26)


In [6]:
classifier = WISDMClassifier(in_c=sampeable.shape[0], num_classes=sampeable.num_classes)

trainer = ClassifierTrainer(
    classifier=classifier,
    train_data=sampeable,
    val_data=val_sampeable,
)

In [7]:
state_dict = trainer.train(
    num_epochs=15,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
    patience=5,
)

2025-10-22 11:03:13,292 - flow-matching - INFO - Training model with size: 0.453 MiB
Epoch 0/15: 100%|██████████| 500/500 [01:12<00:00,  6.93it/s, train_loss=0.556558]
2025-10-22 11:04:25,963 - flow-matching - INFO - val loss: 0.878000020980835, best val loss: 0.878000020980835
Epoch 1/15: 100%|██████████| 500/500 [01:11<00:00,  6.96it/s, train_loss=0.221293]
2025-10-22 11:05:38,313 - flow-matching - INFO - val loss: 0.9179999828338623, best val loss: 0.9179999828338623
Epoch 2/15: 100%|██████████| 500/500 [01:12<00:00,  6.92it/s, train_loss=0.156564]
2025-10-22 11:06:51,138 - flow-matching - INFO - val loss: 0.9419999718666077, best val loss: 0.9419999718666077
Epoch 3/15: 100%|██████████| 500/500 [01:13<00:00,  6.82it/s, train_loss=0.127365]
2025-10-22 11:08:04,959 - flow-matching - INFO - val loss: 0.9539999961853027, best val loss: 0.9539999961853027
Epoch 4/15: 100%|██████████| 500/500 [01:13<00:00,  6.83it/s, train_loss=0.096835]
2025-10-22 11:09:18,655 - flow-matching - INFO - v

KeyboardInterrupt: 

In [8]:
torch.save(classifier.state_dict(), "./models/classifier.pt")